# Task 12 · E-Sign Integration & Tamper-Evidence

# Resume / Job Description Parsing v0

## Objective

The objective of this notebook is to build the first version (v0) of a Resume and Job Description parser.

The parser extracts structured skills from student resumes and job descriptions so that they can be used for downstream matching and recommendation.

## Deliverables

- Load real datasets
- Resume Parsing v0
- Job Description Parsing v0
- Structured Skill Extraction
- Live Verification
- One End-to-End Example
- Parsing Evaluation
- Edge Case Handling
- Business Interpretation

**Definition of Done:** Parsing v0 produces structured skills.

# 1. Import Libraries

The following libraries are required for parsing, preprocessing and data manipulation.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns",None)
pd.set_option("display.width",150)

# 2. Load Datasets

Three datasets are loaded:

- students.csv
- jobs.csv
- matches.csv

These datasets are used to simulate resume parsing and job description parsing.

In [2]:
students = pd.read_csv("../datasets/students.csv")
jobs = pd.read_csv("../datasets/jobs.csv")
matches = pd.read_csv("../datasets/matches.csv")

In [3]:
print("="*70)
print("Students Dataset")
print("="*70)
display(students.head())

print("="*70)
print("Jobs Dataset")
print("="*70)
display(jobs.head())

print("="*70)
print("Matches Dataset")
print("="*70)
display(matches.head())

Students Dataset


,student_id,skills,internship_months,education_level,certifications,preferred_role,location
0,1,"Python:85,SQL:75,Excel:70,Pandas:80",18,BTech,"Python,SQL",Data Analyst,Pune
1,2,"Java:80,Spring:75,SQL:65,Git:70",24,BE,Java,Backend Developer,Mumbai
2,3,"Python:90,ML:85,TensorFlow:75,SQL:70",12,MCA,ML,ML Engineer,Bangalore
3,4,"Excel:85,SQL:60,PowerBI:80",14,BTech,PowerBI,BI Analyst,Pune
4,5,"JavaScript:85,React:80,HTML:90,CSS:85",16,BE,Web,Frontend Developer,Hyderabad


Jobs Dataset


,job_id,company_name,job_title,required_skills,min_experience_years,job_type,location
0,101,TechNova,Data Analyst,"Python:70,SQL:60,Excel:50",1,Hybrid,Pune
1,102,CodeWorks,Backend Developer,"Java:70,Spring:65,SQL:60",2,Remote,Mumbai
2,103,AI Labs,ML Engineer,"Python:80,ML:70,TensorFlow:60",1,Hybrid,Bangalore
3,104,DataVision,BI Analyst,"Excel:70,SQL:60,PowerBI:70",1,Onsite,Pune
4,105,WebCraft,Frontend Developer,"JavaScript:70,React:70,HTML:70",1,Remote,Hyderabad


Matches Dataset


,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,label
0,1,101,3,1.000,2.0,1
1,1,102,1,0.333,1.0,0
2,1,103,1,0.333,2.0,0
3,1,104,2,0.667,2.0,1
4,1,105,0,0.000,2.0,0


In [4]:
print("="*70)
print("DATASET SUMMARY")
print("="*70)

print(f"Students : {students.shape}")
print(f"Jobs     : {jobs.shape}")
print(f"Matches  : {matches.shape}")

print("\nMissing Values")

print(students.isnull().sum())

print()

print(jobs.isnull().sum())

print()

print(matches.isnull().sum())

DATASET SUMMARY
Students : (20, 7)
Jobs     : (9, 7)
Matches  : (180, 6)

Missing Values
student_id           0
skills               0
internship_months    0
education_level      0
certifications       1
preferred_role       0
location             0
dtype: int64

job_id                  0
company_name            0
job_title               0
required_skills         0
min_experience_years    0
job_type                0
location                0
dtype: int64

student_id             0
job_id                 0
skill_overlap_count    0
skill_overlap_ratio    0
experience_gap         0
label                  0
dtype: int64


# 3. Resume Parsing v0

The student resume skills are stored in the following format:

Python:85,SQL:75,Excel:70

The parser converts this string into a structured dictionary containing:

- Skill Name
- Skill Score

This structured representation is easier to use for recommendation systems.

In [5]:
def parse_resume_skills(skill_text):

    skills = {}

    if pd.isna(skill_text):
        return skills

    items = skill_text.split(",")

    for item in items:

        if ":" in item:

            skill, score = item.split(":")

            skills[skill.strip()] = int(score)

    return skills

# 4. Job Description Parsing v0

The required skills for each job are stored in the format:

Python:70,SQL:60,Excel:50

The parser extracts the required skill names and their expected proficiency.

In [6]:
def parse_job_skills(skill_text):

    skills = {}

    if pd.isna(skill_text):
        return skills

    items = skill_text.split(",")

    for item in items:

        if ":" in item:

            skill, score = item.split(":")

            skills[skill.strip()] = int(score)

    return skills

In [7]:
students["parsed_resume"] = students["skills"].apply(parse_resume_skills)

jobs["parsed_job"] = jobs["required_skills"].apply(parse_job_skills)

display(
    students[
        [
            "student_id",
            "parsed_resume"
        ]
    ].head()
)

display(
    jobs[
        [
            "job_id",
            "parsed_job"
        ]
    ].head()
)

,student_id,parsed_resume
0,1,"{'Python': 85, 'SQL': 75, 'Excel': 70, 'Pandas..."
1,2,"{'Java': 80, 'Spring': 75, 'SQL': 65, 'Git': 70}"
2,3,"{'Python': 90, 'ML': 85, 'TensorFlow': 75, 'SQ..."
3,4,"{'Excel': 85, 'SQL': 60, 'PowerBI': 80}"
4,5,"{'JavaScript': 85, 'React': 80, 'HTML': 90, 'C..."


,job_id,parsed_job
0,101,"{'Python': 70, 'SQL': 60, 'Excel': 50}"
1,102,"{'Java': 70, 'Spring': 65, 'SQL': 60}"
2,103,"{'Python': 80, 'ML': 70, 'TensorFlow': 60}"
3,104,"{'Excel': 70, 'SQL': 60, 'PowerBI': 70}"
4,105,"{'JavaScript': 70, 'React': 70, 'HTML': 70}"


# 5. Structured Skill Extraction

After parsing, both resumes and job descriptions contain structured dictionaries.

These structured skills form the foundation for intelligent resume-job matching.

In [8]:
print("="*70)
print("STRUCTURED RESUME SKILLS")
print("="*70)

print(students.loc[0,"parsed_resume"])

print()

print("="*70)
print("STRUCTURED JOB SKILLS")
print("="*70)

print(jobs.loc[0,"parsed_job"])

STRUCTURED RESUME SKILLS
{'Python': 85, 'SQL': 75, 'Excel': 70, 'Pandas': 80}

STRUCTURED JOB SKILLS
{'Python': 70, 'SQL': 60, 'Excel': 50}


# 6. Extract Structured Skills

The parsed dictionaries are converted into separate skill lists.

This makes it easier to compare resume skills with job requirements.

In [9]:
students["resume_skill_names"] = students["parsed_resume"].apply(lambda x: list(x.keys()))
jobs["job_skill_names"] = jobs["parsed_job"].apply(lambda x: list(x.keys()))

display(
    students[
        [
            "student_id",
            "resume_skill_names"
        ]
    ].head()
)

display(
    jobs[
        [
            "job_id",
            "job_skill_names"
        ]
    ].head()
)

,student_id,resume_skill_names
0,1,"[Python, SQL, Excel, Pandas]"
1,2,"[Java, Spring, SQL, Git]"
2,3,"[Python, ML, TensorFlow, SQL]"
3,4,"[Excel, SQL, PowerBI]"
4,5,"[JavaScript, React, HTML, CSS]"


,job_id,job_skill_names
0,101,"[Python, SQL, Excel]"
1,102,"[Java, Spring, SQL]"
2,103,"[Python, ML, TensorFlow]"
3,104,"[Excel, SQL, PowerBI]"
4,105,"[JavaScript, React, HTML]"


# 7. Resume vs Job Skill Comparison

The parser compares the extracted resume skills with the required job skills.

Matching skills are identified automatically.

In [10]:
comparison = matches.merge(
    students[
        [
            "student_id",
            "resume_skill_names"
        ]
    ],
    on="student_id"
).merge(
    jobs[
        [
            "job_id",
            "job_skill_names"
        ]
    ],
    on="job_id"
)

comparison["matched_skills"] = comparison.apply(
    lambda row: list(
        set(row["resume_skill_names"]) &
        set(row["job_skill_names"])
    ),
    axis=1
)

comparison["matched_skill_count"] = comparison["matched_skills"].apply(len)

display(
    comparison[
        [
            "student_id",
            "job_id",
            "matched_skills",
            "matched_skill_count"
        ]
    ].head(10)
)

,student_id,job_id,matched_skills,matched_skill_count
0,1,101,"[SQL, Excel, Python]",3
1,1,102,[SQL],1
2,1,103,[Python],1
3,1,104,"[SQL, Excel]",2
4,1,105,[],0
5,1,106,[],0
6,1,107,[],0
7,1,108,[],0
8,1,109,[SQL],1
9,2,101,[SQL],1


# 8. Parsing Evaluation

The parser is evaluated using:

- Resume Parsing Success
- Job Parsing Success
- Average Skills Extracted

These metrics verify that Parsing v0 is functioning correctly.

In [11]:
resume_success = (
    students["parsed_resume"]
    .apply(len)
    .gt(0)
    .mean()
)

job_success = (
    jobs["parsed_job"]
    .apply(len)
    .gt(0)
    .mean()
)

avg_resume_skills = students["parsed_resume"].apply(len).mean()

avg_job_skills = jobs["parsed_job"].apply(len).mean()

metrics = pd.DataFrame({

    "Metric":[
        "Resume Parsing Success",
        "Job Parsing Success",
        "Average Resume Skills",
        "Average Job Skills"
    ],

    "Value":[
        f"{resume_success:.2%}",
        f"{job_success:.2%}",
        round(avg_resume_skills,2),
        round(avg_job_skills,2)
    ]

})

display(metrics)

,Metric,Value
0,Resume Parsing Success,100.00%
1,Job Parsing Success,100.00%
2,Average Resume Skills,3.25
3,Average Job Skills,3.0


In [12]:
print("="*70)
print("LIVE PARSING METRICS")
print("="*70)

print(f"Resume Parsing Success : {resume_success:.2%}")
print(f"Job Parsing Success    : {job_success:.2%}")
print(f"Average Resume Skills  : {avg_resume_skills:.2f}")
print(f"Average Job Skills     : {avg_job_skills:.2f}")

LIVE PARSING METRICS
Resume Parsing Success : 100.00%
Job Parsing Success    : 100.00%
Average Resume Skills  : 3.25
Average Job Skills     : 3.00


# 9. Live Parsing Demonstration

The following example demonstrates one complete parsing workflow using real sample data.

The walkthrough shows:

- Resume Skills
- Job Skills
- Parsed Output
- Matching Skills

In [13]:
example = comparison.iloc[0]

print("="*70)
print("REAL PARSING EXAMPLE")
print("="*70)

print(f"Student ID : {example['student_id']}")
print(f"Job ID     : {example['job_id']}")

print("\nResume Skills")
print(example["resume_skill_names"])

print("\nJob Skills")
print(example["job_skill_names"])

print("\nMatched Skills")
print(example["matched_skills"])

print(f"\nTotal Matched Skills : {example['matched_skill_count']}")

REAL PARSING EXAMPLE
Student ID : 1
Job ID     : 101

Resume Skills
['Python', 'SQL', 'Excel', 'Pandas']

Job Skills
['Python', 'SQL', 'Excel']

Matched Skills
['SQL', 'Excel', 'Python']

Total Matched Skills : 3


# 10. Parsing Verification

The extracted structured skills are successfully generated from both resumes and job descriptions.

These structured representations can now be used for recommendation, ranking and matching pipelines.

# 11. Failure Handling & Edge Cases

To improve the robustness of Parsing v0, the parser is tested against common edge cases:

- Empty skill string
- Missing values
- Invalid skill format
- Duplicate skills

These tests ensure that the parser remains reliable under different input conditions.

In [14]:
print("="*70)
print("FAILURE HANDLING TESTS")
print("="*70)

# Empty string
print("Empty Input:")
print(parse_resume_skills(""))

# Missing value
print("\nMissing Value:")
print(parse_resume_skills(np.nan))

# Invalid format
print("\nInvalid Format:")
print(parse_resume_skills("Python,SQL,Excel"))

# Duplicate skills
print("\nDuplicate Skills:")
print(parse_resume_skills("Python:80,Python:90,SQL:70"))

print("\n✓ Parser handled all edge cases successfully.")

FAILURE HANDLING TESTS
Empty Input:
{}

Missing Value:
{}

Invalid Format:
{}

Duplicate Skills:
{'Python': 90, 'SQL': 70}

✓ Parser handled all edge cases successfully.


# 12. Parsing Dashboard

The dashboard summarizes the overall performance of Parsing v0.

These metrics demonstrate that structured skills are successfully extracted from resumes and job descriptions.

In [15]:
dashboard = pd.DataFrame({

    "Metric":[
        "Resume Parsing Success",
        "Job Parsing Success",
        "Average Resume Skills",
        "Average Job Skills",
        "Average Matched Skills"
    ],

    "Value":[
        f"{resume_success:.2%}",
        f"{job_success:.2%}",
        round(avg_resume_skills,2),
        round(avg_job_skills,2),
        round(comparison["matched_skill_count"].mean(),2)
    ]

})

display(dashboard)

,Metric,Value
0,Resume Parsing Success,100.00%
1,Job Parsing Success,100.00%
2,Average Resume Skills,3.25
3,Average Job Skills,3.0
4,Average Matched Skills,0.56


# 13. Live Verification

This section verifies that Parsing v0 successfully converts unstructured skill strings into structured representations that can be used for downstream matching.

In [16]:
print("="*70)
print("LIVE VERIFICATION")
print("="*70)

print(f"Students Parsed Successfully : {students.shape[0]}")
print(f"Jobs Parsed Successfully     : {jobs.shape[0]}")
print(f"Resume Parsing Success Rate  : {resume_success:.2%}")
print(f"Job Parsing Success Rate     : {job_success:.2%}")

print("\n✓ Structured skills generated successfully.")
print("✓ Parsing v0 verified on real datasets.")

LIVE VERIFICATION
Students Parsed Successfully : 20
Jobs Parsed Successfully     : 9
Resume Parsing Success Rate  : 100.00%
Job Parsing Success Rate     : 100.00%

✓ Structured skills generated successfully.
✓ Parsing v0 verified on real datasets.


# 14. One Real Example Summary

The table below summarizes one real parsing example using the loaded datasets.

In [17]:
example_summary = pd.DataFrame({

    "Student ID":[example["student_id"]],
    "Job ID":[example["job_id"]],
    "Resume Skills":[", ".join(example["resume_skill_names"])],
    "Job Skills":[", ".join(example["job_skill_names"])],
    "Matched Skills":[", ".join(example["matched_skills"])],
    "Matched Skill Count":[example["matched_skill_count"]]

})

display(example_summary)

,Student ID,Job ID,Resume Skills,Job Skills,Matched Skills,Matched Skill Count
0,1,101,"Python, SQL, Excel, Pandas","Python, SQL, Excel","SQL, Excel, Python",3


# 15. Business Interpretation

Parsing v0 successfully transforms resume and job description skill strings into structured data.

### Benefits

- Enables automated resume-job matching.
- Produces standardized skill representations.
- Improves downstream recommendation accuracy.
- Supports explainable AI matching.
- Forms the foundation for advanced parsing and NLP in future versions.

In [18]:
summary = pd.DataFrame({

    "Component":[
        "Resume Parser",
        "Job Parser",
        "Structured Skills",
        "Matching Ready"
    ],

    "Status":[
        "Completed",
        "Completed",
        "Generated",
        "Yes"
    ]

})

display(summary)

,Component,Status
0,Resume Parser,Completed
1,Job Parser,Completed
2,Structured Skills,Generated
3,Matching Ready,Yes


# 16. Conclusion

This notebook successfully implements Resume and Job Description Parsing v0.

## Key Achievements

- Loaded real datasets.
- Parsed resume skills into structured dictionaries.
- Parsed job description skills into structured dictionaries.
- Compared extracted resume and job skills.
- Evaluated parsing success using quantitative metrics.
- Demonstrated one real end-to-end parsing example.
- Tested edge cases and failure scenarios.
- Verified structured skill extraction using live data.

**Final Result:** **Parsing v0 successfully produces structured skills and is ready for downstream matching.**